In [17]:
!pip install numpy
!pip install qiskit

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\palob\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached rustworkx-0.17.1-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.8.0-py3-none-any.whl.metadata (2.3 kB)
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.1 MB 6.3 MB/s eta 0:00:02
   ----------- ---------------------------- 2.6/9.1 MB 6.9 MB/s eta 0:00:01
   ------------------ --------------------- 4.2/9.1 MB 6.8 MB/s eta 0:00:01
   ------------------------ --------------- 5.5/9.1 MB 6.7 MB/s eta 0:00:01
   ------------------------------- -------- 7.1/9.1 MB 6.7 MB/s eta 0:00:01
   ------------------------------------ --- 8.4/9.1 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 9.1/9.1 MB 6.6 MB/s eta 0:00:00
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
U


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\palob\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [18]:
# Constants for color print
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

def print_matrix(matrix):

    rows = matrix.shape[0]
    columns = matrix.shape[1]

    number_size = max([len(f"{matrix[r, c]:.3f}") for r in range(rows) for c in range(columns)])

    for r in range(rows):
        for c in range(columns):
            element = ""
            non_zero = False

            if matrix[r, c] != 0:
                element = f"{matrix[r, c]:.3f}"
                non_zero = True
                # print(bcolors.FAIL + f"{matrix[r, c]:.2f}" + bcolors.ENDC, end = " "*4)

            elif matrix[r, c] == 0:
                element = "0"
                # print("0", end = " "*4)

            spacing_left = ((number_size - len(element)) // 2)
            spacing_rigth = spacing_left + (number_size - len(element)) % 2

            element = " "*spacing_left + element + " "*spacing_rigth
            print(element, end = " "*4)
            


        print("\n", end = "") 

In [41]:
import numpy as np
from pathlib import Path
from two_level_decomposition import *
dimensions = 16

# Generate random Hermitian matrix
H = 2 * np.random.rand(dimensions, dimensions) - np.ones((dimensions, dimensions))
H = H + 2j * (2 * np.random.rand(dimensions, dimensions) - np.ones((dimensions, dimensions)))
H = H + H.conj().T  # Hermitian matrix

_, U = np.linalg.eigh(H)  # U is unitary (columns are eigenvectors)

matrices, matrices_json = two_level_decomposition(U, dimensions)

A = np.eye(dimensions, dtype=complex)
for i in range(1, len(matrices)):   # skip index 0 (None placeholder)
    if matrices[i] is not None:
        A = matrices[i] @ A
        

# Check if A† = U1† * U2† * ... is equal to U
print(np.all(np.abs(A.conj().T - U) < 1e-10))

True


In [42]:
def unitary_to_json(U: np.ndarray, path: str | Path) -> Path:
    """
    Exporta la matriz unitaria a JSON alternando partes real e imaginaria.

    Formato::

        [
          [[a11_real, a11_imag], [a12_real, a12_imag]],
          [[a21_real, a21_imag], [a22_real, a22_imag]]
        ]
    """
    output = Path(path)
    payload = np.stack((np.round(U.real, 10), np.round(U.imag, 10)), axis=-1).tolist()
    output.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return output


In [43]:
unitary_to_json(U, "U1_unitary_cpiada.json")


WindowsPath('U1_unitary_cpiada.json')

In [44]:
print_matrix(matrices[5])

0.85-0.00j          0              0              0              0         0.04-0.52j          0              0              0              0              0              0              0              0              0              0         
     0         1.00+0.00j          0              0              0              0              0              0              0              0              0              0              0              0              0              0         
     0              0         1.00+0.00j          0              0              0              0              0              0              0              0              0              0              0              0              0         
     0              0              0         1.00+0.00j          0              0              0              0              0              0              0              0              0              0              0              0         
     0              0              0

In [45]:
matrices_json[4]

{'states': [0, 5],
 'matrix': array([[ 0.85135306-1.07957500e-18j,  0.03716316-5.23275133e-01j],
        [ 0.03716316+5.23275133e-01j, -0.85135306-1.07957500e-18j]])}

In [46]:
for k, matrix_json in enumerate(matrices_json):
    print(f"U{k + 1}:")
    print(f"States in which the matrix act: {matrix_json['states']}")
    print(f"2 by 2 non-trivial block:")
    print_matrix( matrix_json["matrix"])
    print(f"Check if non-trivial block is unitary: {is_unitary(matrix_json["matrix"])}")
    print()

U1:
States in which the matrix act: [0, 1]
2 by 2 non-trivial block:
-0.91+0.00j    -0.38+0.15j    
-0.38-0.15j    0.91-0.00j     
Check if non-trivial block is unitary: True

U2:
States in which the matrix act: [0, 2]
2 by 2 non-trivial block:
0.74+0.00j     0.54-0.41j     
0.54+0.41j     -0.74+0.00j    
Check if non-trivial block is unitary: True

U3:
States in which the matrix act: [0, 3]
2 by 2 non-trivial block:
0.91+0.00j     -0.36-0.19j    
-0.36+0.19j    -0.91+0.00j    
Check if non-trivial block is unitary: True

U4:
States in which the matrix act: [0, 4]
2 by 2 non-trivial block:
0.97+0.00j     0.16+0.16j     
0.16-0.16j     -0.97+0.00j    
Check if non-trivial block is unitary: True

U5:
States in which the matrix act: [0, 5]
2 by 2 non-trivial block:
0.85-0.00j     0.04-0.52j     
0.04+0.52j     -0.85-0.00j    
Check if non-trivial block is unitary: True

U6:
States in which the matrix act: [0, 6]
2 by 2 non-trivial block:
0.80-0.00j     -0.57-0.17j    
-0.57+0.17j    -0.80

In [47]:
print_matrix(U)

-0.21+0.00j    -0.21+0.00j    -0.34+0.00j    0.30+0.00j     -0.29+0.00j    0.10+0.00j     0.30+0.00j     0.20+0.00j     0.11+0.00j     0.17+0.00j     0.45+0.00j     0.27+0.00j     0.27+0.00j     -0.22+0.00j    0.11+0.00j     -0.17+0.00j    
-0.09-0.03j    0.25-0.27j     -0.01+0.17j    -0.00+0.23j    -0.03-0.17j    0.15-0.23j     0.10-0.20j     0.31+0.11j     0.17-0.14j     -0.18-0.06j    -0.07-0.01j    -0.04-0.05j    0.26-0.23j     0.26+0.03j     0.03+0.39j     0.26-0.03j     
0.17+0.13j     -0.14+0.10j    0.22-0.24j     -0.20+0.06j    -0.01-0.07j    0.31-0.01j     0.46-0.07j     0.13-0.17j     -0.26-0.08j    -0.02+0.18j    -0.14+0.13j    0.24+0.08j     -0.10-0.07j    -0.05+0.32j    0.20-0.04j     0.15+0.15j     
-0.12+0.06j    -0.00+0.02j    -0.06+0.20j    0.01+0.14j     -0.26-0.23j    0.38-0.19j     -0.22+0.15j    -0.12-0.04j    -0.08+0.21j    0.01-0.17j     -0.06-0.31j    0.13+0.34j     -0.32+0.07j    0.12+0.19j     0.04+0.08j     -0.19-0.13j    
0.06-0.05j     0.04+0.26j     0.21+0

In [48]:
dict_circuit_abstraction2 = convert_1to2(U, "abstraction2.json")

In [49]:
from convert_2to3 import *

dict_circuit_abstraction3 = convert_2to3(dict_circuit_abstraction2, "abstraction3.json")

In [50]:
from generate_circuits import *

In [55]:
qc2 = json_to_circuit_abstraction2(dict_circuit_abstraction2)
qc2.draw()

»
q_0: ─────o──────────o───────o───────o───────o───────o───────o───────o─────»
          │          │       │       │       │  ┌────┴────┐┌─┴─┐     │     »
q_1: ─────o──────────o───────o───────o───────o──┤ Unitary ├┤ X ├─────■─────»
          │     ┌────┴────┐┌─┴─┐     │     ┌─┴─┐└────┬────┘└─┬─┘     │     »
q_2: ─────o─────┤ Unitary ├┤ X ├─────■─────┤ X ├─────o───────o───────o─────»
     ┌────┴────┐└────┬────┘└─┬─┘┌────┴────┐└─┬─┘     │       │  ┌────┴────┐»
q_3: ┤ Unitary ├─────o───────o──┤ Unitary ├──o───────o───────o──┤ Unitary ├»
     └─────────┘                └─────────┘                     └─────────┘»
«                                                              ┌─────────┐┌───┐»
«q_0: ──o────o───────o───────o────o────o───────o───────o────o──┤ Unitary ├┤ X ├»
«     ┌─┴─┐┌─┴─┐     │     ┌─┴─┐┌─┴─┐  │       │       │  ┌─┴─┐└────┬────┘└─┬─┘»
«q_1: ┤ X ├┤ X ├─────■─────┤ X ├┤ X ├──■───────■───────■──┤ X ├─────o───────o──»
«     └─┬─┘└─┬─┘┌────┴────┐└─┬─┘└─┬─┘┌─┴─┐     │     ┌─┴─┐└─┬─┘     │       │  »
«q_2: ──o────o──┤ Unitary ├──o────o──┤ X ├─────■─────┤ X ├──o───────o───────o──»
«       │    │  └────┬────┘  │    │  └─┬─┘┌────┴────┐└─┬─┘  │       │       │  »
«q_3: ──o────o───────o───────o────o────o──┤ Unitary ├──o────o───────o───────o──»
«                                         └─────────┘                          »
«                ┌───┐┌───┐           ┌───┐┌───┐                     ┌───┐┌───┐»
«q_0: ─────■─────┤ X ├┤ X ├─────■─────┤ X ├┤ X ├──■───────■───────■──┤ X ├┤ X ├»
«          │     └─┬─┘└─┬─┘     │     └─┬─┘└─┬─┘  │       │       │  └─┬─┘└─┬─┘»
«q_1: ─────o───────o────o───────o───────o────o────o───────o───────o────o────o──»
«          │       │    │  ┌────┴────┐  │    │  ┌─┴─┐     │     ┌─┴─┐  │    │  »
«q_2: ─────o───────o────o──┤ Unitary ├──o────o──┤ X ├─────■─────┤ X ├──o────o──»
«     ┌────┴────┐  │    │  └────┬────┘  │    │  └─┬─┘┌────┴────┐└─┬─┘  │    │  »
«q_3: ┤ Unitary ├──o────o───────o───────o────o────o──┤ Unitary ├──o────o────o──»
«     └─────────┘                                    └─────────┘               »
«                ┌───┐┌───┐                     ┌───┐┌───┐                     »
«q_0: ─────■─────┤ X ├┤ X ├──■───────■───────■──┤ X ├┤ X ├──■───────■───────■──»
«     ┌────┴────┐└─┬─┘└─┬─┘┌─┴─┐     │     ┌─┴─┐└─┬─┘└─┬─┘┌─┴─┐     │     ┌─┴─┐»
«q_1: ┤ Unitary ├──o────o──┤ X ├─────■─────┤ X ├──o────o──┤ X ├─────■─────┤ X ├»
«     └────┬────┘  │    │  └─┬─┘     │     └─┬─┘  │    │  └─┬─┘┌────┴────┐└─┬─┘»
«q_2: ─────o───────o────o────o───────o───────o────o────o────o──┤ Unitary ├──o──»
«          │       │    │    │  ┌────┴────┐  │    │    │    │  └────┬────┘  │  »
«q_3: ─────o───────o────o────o──┤ Unitary ├──o────o────o────o───────o───────o──»
«                               └─────────┘                                    »
«     ┌───┐┌───┐                               ┌───┐                     »
«q_0: ┤ X ├┤ X ├──■────■───────■───────■────■──┤ X ├──o───────o───────o──»
«     └─┬─┘└─┬─┘┌─┴─┐  │       │       │  ┌─┴─┐└─┬─┘  │       │       │  »
«q_1: ──o────o──┤ X ├──■───────■───────■──┤ X ├──o────o───────o───────o──»
«       │    │  └─┬─┘┌─┴─┐     │     ┌─┴─┐└─┬─┘  │  ┌─┴─┐     │     ┌─┴─┐»
«q_2: ──o────o────o──┤ X ├─────■─────┤ X ├──o────o──┤ X ├─────■─────┤ X ├»
«       │    │    │  └─┬─┘┌────┴────┐└─┬─┘  │    │  └─┬─┘┌────┴────┐└─┬─┘»
«q_3: ──o────o────o────o──┤ Unitary ├──o────o────o────■──┤ Unitary ├──■──»
«                         └─────────┘                    └─────────┘     »
«                                                                          »
«q_0: ─────o───────o───────o───────o───────o───────o────o───────o───────o──»
«          │     ┌─┴─┐     │     ┌─┴─┐┌────┴────┐┌─┴─┐  │       │       │  »
«q_1: ─────o─────┤ X ├─────■─────┤ X ├┤ Unitary ├┤ X ├──■───────■───────■──»
«     ┌────┴────┐└─┬─┘     │     └─┬─┘└────┬────┘└─┬─┘┌─┴─┐     │     ┌─┴─┐»
«q_2: ┤ Unitary ├──o───────o───────o───────o───────o──┤ X ├─────■─────┤ X ├»
«     └────┬────┘  │  ┌────┴────┐  │       │       │  └─┬─

In [56]:
qc3 = json_to_circuit_abstraction3(dict_circuit_abstraction3)
qc3.draw()

»
q_0: ──o───────────────o────o───────────────o────o────o───────────────o────o──»
       │               │    │               │    │    │               │    │  »
q_1: ──o───────────────o────o───────────────o────o────o───────────────o────o──»
       │               │    │  ┌─────────┐  │  ┌─┴─┐  │               │  ┌─┴─┐»
q_2: ──o───────────────o────┼──┤ Unitary ├──┼──┤ X ├──■───────────────■──┤ X ├»
       │  ┌─────────┐  │    │  └────┬────┘  │  └─┬─┘  │  ┌─────────┐  │  └─┬─┘»
q_3: ──┼──┤ Unitary ├──┼────o───────┼───────o────o────┼──┤ Unitary ├──┼────o──»
     ┌─┴─┐└────┬────┘┌─┴─┐┌─┴─┐     │     ┌─┴─┐     ┌─┴─┐└────┬────┘┌─┴─┐     »
q_4: ┤ X ├─────o─────┤ X ├┤ X ├─────o─────┤ X ├─────┤ X ├─────o─────┤ X ├─────»
     └───┘           └───┘└───┘           └───┘     └───┘           └───┘     »
«                                                                              »
«q_0: ──o───────────────o────o────o───────────────o────o────o────o─────────────»
«       │  ┌─────────┐  │  ┌─┴─┐  │               │  ┌─┴─┐┌─┴─┐  │             »
«q_1: ──┼──┤ Unitary ├──┼──┤ X ├──■───────────────■──┤ X ├┤ X ├──■─────────────»
«       │  └────┬────┘  │  └─┬─┘  │               │  └─┬─┘└─┬─┘  │  ┌─────────┐»
«q_2: ──o───────┼───────o────o────o───────────────o────o────o────┼──┤ Unitary ├»
«       │       │       │    │    │  ┌─────────┐  │    │    │    │  └────┬────┘»
«q_3: ──o───────┼───────o────o────┼──┤ Unitary ├──┼────o────o────o───────┼─────»
«     ┌─┴─┐     │     ┌─┴─┐     ┌─┴─┐└────┬────┘┌─┴─┐          ┌─┴─┐     │     »
«q_4: ┤ X ├─────o─────┤ X ├─────┤ X ├─────o─────┤ X ├──────────┤ X ├─────o─────»
«     └───┘           └───┘     └───┘           └───┘          └───┘           »
«                                                             ┌─────────┐     »
«q_0: ──o────o────o────o────o───────────────o────o────o───────┤ Unitary ├─────»
«       │  ┌─┴─┐┌─┴─┐  │    │               │    │  ┌─┴─┐     └────┬────┘     »
«q_1: ──■──┤ X ├┤ X ├──■────■───────────────■────■──┤ X ├──o───────┼───────o──»
«       │  └─┬─┘└─┬─┘┌─┴─┐  │               │  ┌─┴─┐└─┬─┘  │       │       │  »
«q_2: ──┼────o────o──┤ X ├──■───────────────■──┤ X ├──o────o───────┼───────o──»
«       │    │    │  └─┬─┘  │  ┌─────────┐  │  └─┬─┘  │    │       │       │  »
«q_3: ──o────o────o────o────┼──┤ Unitary ├──┼────o────o────o───────┼───────o──»
«     ┌─┴─┐               ┌─┴─┐└────┬────┘┌─┴─┐          ┌─┴─┐     │     ┌─┴─┐»
«q_4: ┤ X ├───────────────┤ X ├─────o─────┤ X ├──────────┤ X ├─────o─────┤ X ├»
«     └───┘               └───┘           └───┘          └───┘           └───┘»
«     ┌───┐                     ┌───┐┌───┐                     ┌───┐┌───┐     »
«q_0: ┤ X ├──■───────────────■──┤ X ├┤ X ├──■───────────────■──┤ X ├┤ X ├──■──»
«     └─┬─┘  │               │  └─┬─┘└─┬─┘  │               │  └─┬─┘└─┬─┘  │  »
«q_1: ──o────o───────────────o────o────o────o───────────────o────o────o────o──»
«       │    │               │    │    │    │  ┌─────────┐  │    │    │  ┌─┴─┐»
«q_2: ──o────o───────────────o────o────o────┼──┤ Unitary ├──┼────o────o──┤ X ├»
«       │    │  ┌─────────┐  │    │    │    │  └────┬────┘  │    │    │  └─┬─┘»
«q_3: ──o────┼──┤ Unitary ├──┼────o────o────o───────┼───────o────o────o────o──»
«          ┌─┴─┐└────┬────┘┌─┴─┐          ┌─┴─┐     │     ┌─┴─┐               »
«q_4: ─────┤ X ├─────o─────┤ X ├──────────┤ X ├─────o─────┤ X ├───────────────»
«          └───┘           └───┘          └───┘           └───┘               »
«                               ┌───┐┌───┐                     ┌───┐┌───┐     »
«q_0: ──■───────────────■────■──┤ X ├┤ X ├──■───────────────■──┤ X ├┤ X ├──■──»
«       │               │    │  └─┬─┘└─┬─┘  │  ┌─────────┐  │  └─┬─┘└─┬─┘┌─┴─┐»
«q_1: ──o───────────────o────o────o────o────┼──┤ Unitary ├──┼────o────o──┤ X ├»
«       │               │  ┌─┴─┐  │    │    │  └────┬────┘  │    │    │  └─┬─┘»
«q_2: ──■───────────────■──┤ X ├──o────o────o───────┼───────o────o────o────o──»
«       │  ┌─────────┐  │  └─┬─┘  │    │    │       │       │    │ 